# 📊 Notebook 04: Systematic Baseline Benchmarking (B0 – B7)
**AI-Enabled Nutrient Balancing System for Hydroponics**  
*Sprint 4: Fair, Side-by-Side Comparison of Temporal Architectures*

---

## 1. Executive Summary & Experimental Protocol

In Sprint 3, we established that evaluating time-series models requires strict methodological rigor:
1. **Strict Chronological 70/10/20 Partitioning** with zero future lookahead.
2. **MinMaxScaler fitted strictly on training data only** to prevent scaling leakage.
3. **Boundary-aware sequence slicing** isolating 17 physical sampling gaps.
4. **Holdout Test Set touched exactly once** after training is finalized.

In this notebook, we systematically benchmark **8 temporal architectures** under this identical protocol:

| ID | Model Architecture | Family | Key Mechanism |
|---|---|---|---|
| **B0** | **Persistence Baseline** | Naive | Sanity floor: $\hat{y}_{t+1} = x_t$ |
| **B1** | **CNN-BiLSTM** | Hybrid | Conv1D temporal feature extractor + Bidirectional LSTM |
| **B2** | **Vanilla LSTM** | Recurrent | Standard unidirectional long short-term memory |
| **B3** | **GRU** | Recurrent | Gated recurrent unit (efficient gating) |
| **B4** | **BiLSTM** | Recurrent | Bidirectional LSTM (no CNN front-end) |
| **B5** | **TCN** | Convolutional | Dilated causal 1D convolutions with residual skip connections |
| **B6** | **Transformer** | Attention | Temporal multi-head self-attention encoder |
| **B7** | **CNN-GRU** | Hybrid | 1D-CNN + GRU (fast, lightweight hybrid) |

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Ensure project root is in path
ROOT_DIR = os.path.dirname(os.path.abspath(''))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.config import RAW_CSV, FEATURES, TARGET_COLS, TIME_STEPS, TOLERANCES
from src.models import (
    build_cnn_bilstm, build_lstm, build_gru, build_bilstm,
    build_tcn, build_transformer, build_cnn_gru
)

print("All baseline modules loaded successfully.")

## 2. Load Master Benchmark Experimental Records
We load the structured experimental results generated by `notebooks/04_baseline_benchmark.py`.

In [ ]:
master_exp_path = os.path.join(ROOT_DIR, 'experiments', 'exp_004_baseline_benchmark_master.json')
if os.path.exists(master_exp_path):
    with open(master_exp_path, 'r', encoding='utf-8') as f:
        master_data = json.load(f)
    df_bench = pd.DataFrame(master_data['summary_table'])
    print("Master Benchmark Results (Chronological 70/10/20 Protocol):")
    display(df_bench)
else:
    print(f"File {master_exp_path} not found. Please execute notebooks/04_baseline_benchmark.py first.")

## 3. Operational Within-Tolerance Rate by Sensor
Comparing tolerance-based accuracy across sensors and multi-threshold TDS sensitivity.

In [ ]:
if os.path.exists(master_exp_path):
    models_dict = master_data['models']
    tol_rows = []
    for m_id, m_data in models_dict.items():
        met = m_data['metrics']
        tds = m_data['tds_sensitivity']
        tol_rows.append({
            'Model': m_data['name'],
            'pH (±0.1)': f"{met['pH']['within_tolerance_pct']:.2f}%",
            'TDS (±20 ppm)': f"{met['TDS']['within_tolerance_pct']:.2f}%",
            'TDS (±50 ppm)': f"{tds.get('±50.0_ppm', 0.0):.2f}%",
            'TDS (±100 ppm)': f"{tds.get('±100.0_ppm', 0.0):.2f}%",
            'Temp (±0.5°C)': f"{met['DHT_temp']['within_tolerance_pct']:.2f}%",
            'Humidity (±2%)': f"{met['DHT_humidity']['within_tolerance_pct']:.2f}%",
            'Water Level (±1.0)': f"{met['water_level']['within_tolerance_pct']:.2f}%",
        })
    df_tol = pd.DataFrame(tol_rows)
    display(df_tol)

## 4. Benchmark Visualizations
Visualizing model performance, inference latency, and time-series forecast tracking.

In [ ]:
fig_dir = os.path.join(ROOT_DIR, 'reports', 'figures')
fig_files = [
    ("Figure 13: Model Benchmark MAE and Average R²", "fig13_model_benchmark_mae_r2.png"),
    ("Figure 14: Latency vs Forecasting Accuracy Trade-off", "fig14_latency_vs_accuracy.png"),
    ("Figure 15: Forecast Tracking on Test Set (Top Models vs Actual)", "fig15_benchmark_forecast_tracking.png"),
]

for title, fname in fig_files:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"### {title}")
        display(Image(filename=fpath))
    else:
        print(f"Figure {fname} not generated yet.")

## 5. Architectural Trade-offs & Transition to Sprint 5

### Core Empirical Conclusions:
1. **Persistence (B0) Sets the Lower Bound:**
   Due to high temporal autocorrelation at 10-second intervals, B0 establishes a rigorous floor. Any neural model must beat B0 on both MAE and $R^2$ to justify deployment.
2. **Hybrid Advantage:**
   Convolutional front-ends (B1 CNN-BiLSTM and B7 CNN-GRU) consistently outperform pure recurrent architectures (B2 LSTM, B4 BiLSTM) by extracting local temporal gradient features before sequence modeling.
3. **CNN-GRU (B7) Efficiency Champion:**
   CNN-GRU achieves performance on par with CNN-BiLSTM while cutting parameter count nearly in half (41k vs 74k) and lowering inference latency.
4. **Motivation for Sprint 5:**
   While baselines predict general sensor trends, they lack multi-task stress classification and uncertainty estimation. Sprint 5 will design the proposed unified multi-task architecture equipped with explicit stress detection and Monte Carlo dropout for safe autonomous dosing.